# Chunker

In [2]:
# "Limpiamos" MongoDB tras el último chunker ejecutado

from pymongo import MongoClient
import os


uri = os.environ.get('cs_mongo')
client = MongoClient(uri)
db = client["rag"]
col = db["documents"]

# Borramos todo para empezar de cero
x = col.delete_many({})
print(f"Se han eliminado {x.deleted_count} documentos antiguos. La colección está limpia.")

Se han eliminado 2126 documentos antiguos. La colección está limpia.


In [3]:

#  INGESTA DE LIBRERIAS 
import os #La cargamos antes, pero la incluimos por si se comenta la celda anterior.
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mongodb import MongoDBAtlasVectorSearch
from pymongo import MongoClient
import PyPDF2
import docx
import datetime


C:\Users\alejandro.casares\rag_env\Lib\site-packages\langchain_core\_api\deprecation.py:27: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [4]:

# 1. CONFIGURACIÓN

# Usamos un modelo multilingüe, que entienda Español (los pdfs están en ese idioma)
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MONGO_URI = os.environ.get('cs_mongo')
DB_NAME = "rag"
COL_NAME = "documents"
DOCS_PATH = os.environ.get('pathdoc') or "RUTA_A_TUS_PDFS"

print(f"Configurando modelo de embeddings: {MODEL_NAME}")

Configurando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [5]:

# 2. FUNCIONES
def extract_text(file_path: str) -> str:
    text = ""
    try:
        if file_path.lower().endswith(".pdf"):
            reader = PyPDF2.PdfReader(file_path)
            for page in reader.pages:
                text += page.extract_text() + "\n"
        elif file_path.lower().endswith(".docx"):
            document = docx.Document(file_path)
            text = "\n".join([p.text for p in document.paragraphs])
        
        # Limpieza básica: quitar excesos de espacios y saltos de línea raros
        text = text.replace('\xa0', ' ').replace('  ', ' ')
        return text
    except Exception as e:
        print(f"Error leyendo {file_path}: {e}")
        return ""

def procesar_y_cargar():
    # Inicializamos el modelo de embedings
    # Lo cargamos con LangChain, que gestiona la vectorización automáticamente
    embeddings = HuggingFaceEmbeddings(
        model_name=MODEL_NAME,
        model_kwargs={'device': 'cpu'}, # Forzamos CPU
        encode_kwargs={'normalize_embeddings': True} # Mejora la búsqueda
    )


In [9]:
#Inicializamos el modelo de embedings
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={'device': 'cpu'},  # Forzamos uso de CPU
    encode_kwargs={'normalize_embeddings': True} # Normaliza para mejor búsqueda
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:

# 3. INICIAR LA BASE DE DATOS VECTORIAL

print(" Conectando con MongoDB Atlas...")

vectorstore = MongoDBAtlasVectorSearch.from_connection_string(
    connection_string=MONGO_URI,
    namespace=f"{DB_NAME}.{COL_NAME}",
    embedding=embeddings,
    index_name="vector_index" 
)

print(" Conexión con VectorStore establecida.")



 Conectando con MongoDB Atlas...
 Conexión con VectorStore establecida.


In [13]:

# 4. SPLITTER 
    # Corta por párrafos (\n\n), luego líneas (\n), luego puntos, luego espacios.

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,      # Trozos de 500 caracteres
        chunk_overlap=50,    # Solapamiento para no cortar ideas entre trozos
        separators=["\n\n", "\n", ".", " ", ""]
    )

# 5. BUCLE DE PROCESAMIENTO
total_chunks = 0
    
for root, _, files in os.walk(DOCS_PATH):
    for fname in files:
        if not fname.lower().endswith((".pdf", ".docx")):
            continue
                
        path = os.path.join(root, fname)
        print(f" Procesando: {fname}...")
            
        # A. Extrae texto
        raw_text = extract_text(path)
        if not raw_text: continue

        # B. Crear Chunks 
        docs = text_splitter.create_documents(
            texts=[raw_text], 
            metadatas=[{"filename": fname, "topic": os.path.basename(root)}]
        )
            
        # C. Subir a MongoDB
        # add_documents calcula los vectores y los sube a la base de datos vectorial
        vectorstore.add_documents(docs)
            
        n_chunks = len(docs)
        total_chunks += n_chunks
        print(f"   Guardados {n_chunks} fragmentos.")

print(f"\n Proceso terminado! Total fragmentos en MongoDB: {total_chunks}")

# EJECUTAR
if __name__ == "__main__":
    procesar_y_cargar()

 Procesando: Adam Smith - La riqueza de las naciones.pdf...


unknown widths : 
[0, IndirectObject(8543, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(8538, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(8533, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(8523, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(8518, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(8513, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(8503, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5712, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5757, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5757, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5712, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5850, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5757, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5712, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5712, 0, 1585490946976)]
unknown widths : 
[0, IndirectObject(5757, 0, 1585490946976)]
unknown 

   Guardados 3245 fragmentos.
 Procesando: David Ricardo - Principios de Economía Política y Tributación.pdf...
   Guardados 101 fragmentos.
 Procesando: Teoría general del empleo, el interés y el dinero (John M. Keynes).pdf...


   Guardados 1559 fragmentos.

 Proceso terminado! Total fragmentos en MongoDB: 4905


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
